In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

img_size = (224, 224)
batch_size = 32

# 1. ชุด Train: ทำ Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2. ชุด Val และ Test: ห้ามทำ Augmentation (ใช้ภาพจริงเพื่อวัดผล)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("กำลังโหลดข้อมูล Train...")
train_generator = train_datagen.flow_from_directory(
    'train',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

print("กำลังโหลดข้อมูล Validation...")
val_generator = val_test_datagen.flow_from_directory(
    'validation',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("กำลังโหลดข้อมูล Test...")
test_generator = val_test_datagen.flow_from_directory(
    'test',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# บันทึก Label ไว้ใช้ในหน้าเว็บ
labels = {v: k for k, v in train_generator.class_indices.items()}
with open('class_labels.json', 'w') as f:
    json.dump(labels, f)
print(f"เตรียมข้อมูลสำเร็จ! พบทั้งหมด {len(labels)} คลาส")

กำลังโหลดข้อมูล Train...
Found 3115 images belonging to 36 classes.
กำลังโหลดข้อมูล Validation...
Found 351 images belonging to 36 classes.
กำลังโหลดข้อมูล Test...
Found 359 images belonging to 36 classes.
เตรียมข้อมูลสำเร็จ! พบทั้งหมด 36 คลาส


In [4]:
from tensorflow.keras import layers, models

# 1. โหลด MobileNetV2 มาเป็นฐาน (ตัดส่วนหัวออก)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # แช่แข็งน้ำหนักเดิมไว้ก่อน จะได้เทรนไวๆ

# 2. สร้างโครงสร้างโมเดลของเราต่อยอด
num_classes = len(train_generator.class_indices)
nn_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), # ป้องกัน Overfitting
    layers.Dense(num_classes, activation='softmax')
])

nn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 3. เริ่มเทรนโมเดล
print("กำลังเทรนโมเดล Neural Network...")
history = nn_model.fit(
    train_generator,
    epochs=10, # ลองเทรน 10 รอบดูก่อนว่าแม่นไหม
    validation_data=val_generator
)

# 4. สอบปลายภาคด้วย Test Set
print("กำลังทดสอบความแม่นยำด้วย Test Set...")
test_loss, test_acc = nn_model.evaluate(test_generator)
print(f"ความแม่นยำ Neural Network บน Test Set (ข้อสอบที่ไม่เคยเห็น): {test_acc:.4f}")

# เซฟโมเดล
nn_model.save('nn_model.h5')
print("เซฟไฟล์ nn_model.h5 สำเร็จพร้อมเอาไปขึ้นเว็บ!")

กำลังเทรนโมเดล Neural Network...
Epoch 1/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 88s 868ms/step - accuracy: 0.4058 - loss: 2.2368 - val_accuracy: 0.8148 - val_loss: 0.6499
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 84s 858ms/step - accuracy: 0.6693 - loss: 1.1258 - val_accuracy: 0.8860 - val_loss: 0.4164
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 84s 858ms/step - accuracy: 0.7201 - loss: 0.9148 - val_accuracy: 0.9003 - val_loss: 0.3564
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 86s 876ms/step - accuracy: 0.7650 - loss: 0.7496 - val_accuracy: 0.9088 - val_loss: 0.2982
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 85s 869ms/step - accuracy: 0.7865 - loss: 0.6739 - val_accuracy: 0.9174 - val_loss: 0.2809
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 89s 906ms/step - accuracy: 0.8061 - loss: 0.6057 - val_accuracy: 0.9259 - val_loss: 0.2392
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 91s 927ms/step - accuracy: 0.8173 - loss: 0.5526 - val_accuracy: 0.9288 - val_loss: 0.2322
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 84s 856ms/step - accuracy:

ความแม่นยำ Neural Network บน Test Set (ข้อสอบที่ไม่เคยเห็น): 0.9499
เซฟไฟล์ nn_model.h5 สำเร็จพร้อมเอาไปขึ้นเว็บ!
